<a href="https://colab.research.google.com/github/megluc/waymo-project/blob/main/VGG_implementation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q tensorflow

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize
x_train, x_test = x_train / 255.0, x_test / 255.0

# Combine then split
X = np.concatenate([x_train, x_test], axis=0)
Y = np.concatenate([y_train, y_test], axis=0)

x_train, x_test, y_train, y_test = train_test_split(
    X, Y, train_size=0.70, random_state=42
)

# Convert to tf.data
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_dataset  = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Resize to 224x224
def process_images(image, label):
    image = tf.image.resize(image, (224,224))
    return image, label

train_dataset = train_dataset.map(process_images).batch(32)
test_dataset  = test_dataset.map(process_images).batch(32)

In [ ]:
def vgg_block(num_convs, num_channels):
    block = tf.keras.Sequential()

    for _ in range(num_convs):
        block.add(layers.Conv2D(num_channels, (3,3), padding='same', activation='relu'))
        block.add(layers.BatchNormalization())

    block.add(layers.MaxPooling2D((2,2)))
    return block

In [ ]:
# vgg model building
def build_vgg(input_shape=(224,224,3), num_classes=10):

    model = tf.keras.Sequential()

    # initial layer
    model.add(layers.Conv2D(64, (3,3), padding='same',
                            activation='relu',
                            input_shape=input_shape))

    # VGG architecture
    conv_arch = (
        (1, 64),
        (1, 128),
        (2, 256),
        (2, 512),
        (2, 512)
    )

    for num_convs, num_channels in conv_arch:
        model.add(vgg_block(num_convs, num_channels))

    # fully connected layers
    model.add(layers.Flatten())
    model.add(layers.Dense(4096, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(4096, activation='relu'))
    model.add(layers.Dropout(0.5))

    # output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

In [ ]:
model_vgg = build_vgg()

model_vgg.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_vgg.summary()

In [ ]:
#training the model
history_vgg = model_vgg.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

In [ ]:
# plot accuracy
plt.plot(history_vgg.history['accuracy'], label='train')
plt.plot(history_vgg.history['val_accuracy'], label='val')
plt.legend()
plt.title("VGG Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# evaluation of model
test_loss, test_acc = model_vgg.evaluate(test_dataset)
print("Test Accuracy:", test_acc)